# Full Reproducibility Notebook

This notebook is the reviewer-facing rerun path for the deepfake speech detection paper. It does not merely read committed result files: each experiment cell checks for its raw inputs and cached intermediates, regenerates missing cache/data by calling the original repository scripts, and records a full traceback when an external dependency is unavailable.

The notebook is intentionally explicit about long-running steps. Re-running everything from cold caches may download Hugging Face datasets, compute WavLM embeddings, and score multiple detectors. Expect GPU-backed runs to take substantially longer than the lightweight demo notebook.


## 0. Environment setup

Reproduced with **Python 3.12** on a **CUDA 12.9** GPU (NVIDIA L4, 23 GB). Full pinned
dependency list is in `requirements.txt`.

**One-time setup** — run the code cell below, or equivalently in a shell before launching Jupyter:

```bash
python -m pip install torch==2.8.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu129
python -m pip install -r requirements.txt
```

- **Different CUDA / CPU-only:** pick the matching `--index-url` from
  <https://pytorch.org/get-started/locally/>; keep the pinned `2.8.0` version (this is what the
  checkpoints and cached results were produced with).
- **Hugging Face token** — dataset downloads (MLAAD, In-the-Wild, ASVspoof) may need auth.
  Set it before running: `export HF_TOKEN=hf_xxx` (or `huggingface-cli login`).
- **Checkpoints** — frozen detector checkpoints are **not** downloaded automatically; upload them
  into `models/good_models/` (§2 lists the exact filenames and reports any that are missing).

In [ ]:
# ── Environment setup / verification (idempotent — safe to re-run) ────────────
import importlib, subprocess, sys, os
CORE = ['torch','torchaudio','transformers','datasets','huggingface_hub','soundfile',
        'einops','sklearn','pandas','numpy','scipy','pytorch_lightning','matplotlib','pyarrow']
def _missing(mods):
    out=[]
    for m in mods:
        try: importlib.import_module(m)
        except ImportError: out.append(m)
    return out

miss = _missing(CORE)
if miss:
    print('Installing reproducibility environment (missing:', miss, ')...')
    subprocess.check_call([sys.executable,'-m','pip','install',
        'torch==2.8.0','torchaudio==2.8.0','--index-url','https://download.pytorch.org/whl/cu129'])
    subprocess.check_call([sys.executable,'-m','pip','install','-r','requirements.txt'])
    importlib.invalidate_caches()
else:
    print('Core packages already present.')

import torch, torchaudio
print('torch', torch.__version__, '| torchaudio', torchaudio.__version__,
      '| CUDA available:', torch.cuda.is_available())
still = _missing(CORE)
print('All core imports OK.' if not still else f'STILL MISSING (re-run this cell): {still}')
if not os.environ.get('HF_TOKEN'):
    print('NOTE: HF_TOKEN not set. If dataset downloads fail with auth errors, set it and re-run:')
    print('      export HF_TOKEN=hf_xxxxxxxx   (or run: huggingface-cli login)')

## 0. Configuration and Notebook Runner

Set the toggles below before running all cells. `FORCE_REBUILD=False` means a cell reuses valid existing artifacts; set it to `True` when you want a cold regeneration. `STOP_ON_FAILURE=False` lets the notebook keep going and produce a complete failure report with tracebacks.


In [ ]:
from __future__ import annotations
from pathlib import Path
import contextlib
import hashlib
import importlib
import json
import os
import shutil
import subprocess
import sys
import textwrap
import time
import traceback

ROOT = Path.cwd().resolve()
assert (ROOT / 'experiments').exists(), f'Run this notebook from the repository root, got {ROOT}'

# Reviewer toggles
FORCE_REBUILD = False          # rebuild outputs even if expected artifacts already exist
ALLOW_DOWNLOADS = True         # permit Hugging Face dataset downloads if caches are missing
INSTALL_MISSING_PACKAGES = False
STOP_ON_FAILURE = False        # if False, collect tracebacks and continue
RUN_HEAVY_EXPERIMENTS = True   # if False, run preflight/bootstrap/audits only

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('WANDB_MODE', 'disabled')
os.environ.setdefault('HF_HOME', str(ROOT / 'data' / 'huggingface'))
os.environ.setdefault('HF_DATASETS_CACHE', str(ROOT / 'data' / 'huggingface' / 'datasets'))
os.environ.setdefault('HF_HUB_CACHE', str(ROOT / 'data' / 'huggingface' / 'hub'))
os.environ.setdefault('HF_DATASETS_OFFLINE', '0' if ALLOW_DOWNLOADS else '1')
os.environ.setdefault('HF_HUB_OFFLINE', '0' if ALLOW_DOWNLOADS else '1')

secret = ROOT / 'secret.txt'
if secret.exists():
    token = secret.read_text().strip()
    if token:
        os.environ.setdefault('HF_TOKEN', token)
        os.environ.setdefault('HUGGING_FACE_HUB_TOKEN', token)

RUN_LOG = []
FAILURES = []


def rel(p: Path | str) -> str:
    p = Path(p)
    try:
        return str(p.resolve().relative_to(ROOT))
    except Exception:
        return str(p)


def sha256(path: Path, chunk=1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open('rb') as fh:
        while True:
            b = fh.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def paths_exist(paths) -> bool:
    return all(Path(p).exists() for p in paths)


def record_failure(name: str, exc_text: str):
    FAILURES.append({'step': name, 'traceback': exc_text})
    print(f'\n===== TRACEBACK FOR {name} =====')
    print(exc_text)
    print(f'===== END TRACEBACK FOR {name} =====\n')
    if STOP_ON_FAILURE:
        raise RuntimeError(f'{name} failed; see traceback above')


def run_cmd(name: str, cmd: list[str], expected_outputs=(), required_inputs=(), cwd: Path = ROOT, env_extra=None, timeout=None):
    """Run an original repo command with full stdout/stderr capture and traceback reporting."""
    expected_outputs = [Path(p) for p in expected_outputs]
    required_inputs = [Path(p) for p in required_inputs]
    start = time.time()
    print(f'\n### {name}')
    print('Command:', ' '.join(map(str, cmd)))

    missing_inputs = [p for p in required_inputs if not p.exists()]
    if missing_inputs:
        msg = 'Required inputs are missing before command execution:\n' + '\n'.join(f' - {rel(p)}' for p in missing_inputs)
        record_failure(name, msg)
        RUN_LOG.append({'name': name, 'status': 'missing_inputs', 'seconds': time.time() - start})
        return False

    if expected_outputs and paths_exist(expected_outputs) and not FORCE_REBUILD:
        print('SKIP: expected outputs already exist:')
        for p in expected_outputs:
            print(' -', rel(p))
        RUN_LOG.append({'name': name, 'status': 'skipped_existing', 'seconds': time.time() - start})
        return True

    env = os.environ.copy()
    if env_extra:
        env.update(env_extra)
    try:
        proc = subprocess.run(cmd, cwd=str(cwd), env=env, text=True, capture_output=True, timeout=timeout)
        print(proc.stdout[-12000:])
        if proc.returncode != 0:
            tb = 'Subprocess failed with exit code %s\n\nSTDOUT tail:\n%s\n\nSTDERR tail:\n%s' % (
                proc.returncode, proc.stdout[-12000:], proc.stderr[-12000:])
            record_failure(name, tb)
            RUN_LOG.append({'name': name, 'status': 'failed', 'seconds': time.time() - start})
            return False
        if proc.stderr.strip():
            print('STDERR tail:')
            print(proc.stderr[-4000:])
        missing_outputs = [p for p in expected_outputs if not p.exists()]
        if missing_outputs:
            tb = 'Command completed but expected outputs are missing:\n' + '\n'.join(f' - {rel(p)}' for p in missing_outputs)
            record_failure(name, tb)
            RUN_LOG.append({'name': name, 'status': 'missing_outputs', 'seconds': time.time() - start})
            return False
        RUN_LOG.append({'name': name, 'status': 'ok', 'seconds': time.time() - start})
        return True
    except Exception:
        record_failure(name, traceback.format_exc())
        RUN_LOG.append({'name': name, 'status': 'exception', 'seconds': time.time() - start})
        return False


def run_py(name: str, script: str, expected_outputs=(), required_inputs=(), args=(), env_extra=None, timeout=None):
    return run_cmd(name, [sys.executable, script, *map(str, args)], expected_outputs, required_inputs, ROOT, env_extra, timeout)


def import_report(modules):
    rows = []
    for m in modules:
        try:
            mod = importlib.import_module(m)
            rows.append({'module': m, 'status': 'ok', 'version': getattr(mod, '__version__', '')})
        except Exception as exc:
            rows.append({'module': m, 'status': 'missing', 'version': f'{type(exc).__name__}: {exc}'})
    return rows

print('Repo root:', ROOT)
print('ALLOW_DOWNLOADS:', ALLOW_DOWNLOADS)
print('FORCE_REBUILD:', FORCE_REBUILD)
print('RUN_HEAVY_EXPERIMENTS:', RUN_HEAVY_EXPERIMENTS)
print('HF token present:', bool(os.environ.get('HF_TOKEN')))


## 1. Dependency Preflight

This cell checks the Python packages used by the original scripts. It can optionally install missing packages, but by default it only reports them so the environment remains auditable.


In [ ]:
import pandas as pd
required_modules = [
    'numpy', 'pandas', 'scipy', 'sklearn', 'torch', 'torchaudio', 'transformers',
    'datasets', 'huggingface_hub', 'pytorch_lightning', 'soundfile'
]
dep_df = pd.DataFrame(import_report(required_modules))
display(dep_df)
missing = dep_df[dep_df.status != 'ok']['module'].tolist()
if missing and INSTALL_MISSING_PACKAGES:
    run_cmd('install missing Python packages', [sys.executable, '-m', 'pip', 'install', *missing])
elif missing:
    print('Missing packages were not installed because INSTALL_MISSING_PACKAGES=False:', missing)


## 2. Checkpoint Bootstrap

Original scripts expect checkpoints in several historical paths. Checkpoints are **provided by the reviewer** — there is no automatic download. This cell verifies that the required frozen detector checkpoints are present in `models/good_models/` (it reports any that are missing so you can upload them), then creates the symlinks/copies at the hard-coded paths the analysis scripts expect, and validates that every checkpoint loads.


In [ ]:
import os
import torch

GOOD = ROOT / 'models' / 'good_models'
MODELS = ROOT / 'models'
EXP_CKPTS = ROOT / 'experiments' / 'checkpoints'
GOOD.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(exist_ok=True)
EXP_CKPTS.mkdir(parents=True, exist_ok=True)

# ── Checkpoints are provided by the reviewer — NO automatic download ──────────
# Place the frozen detector checkpoints below into  models/good_models/  before
# running the notebook. They cannot be retrained here (training needs the full
# ASVspoof-2019-LA corpus + GPU-days), so upload them manually. Sizes are ~0.5 GB
# each except robust_goat_seed3.ckpt.
REQUIRED_CHECKPOINTS = [
    'robust_goat.ckpt',                                        # ASVspoof19 detector, seed 1 (I1 s1)
    'robust_goat_seed3.ckpt',                                  # ASVspoof19 detector, seed 3 (I1 s3)
    'mini_goat-best-epoch=02-val-eer=0.0933.ckpt',            # mini_goat (ASVspoof19 fusion)
    'mlaad_goat-best-epoch=05-val-eer=0.2795.ckpt',          # MLAAD GOAT
    'mlaad_robust_goat.ckpt',                                  # MLAAD robust GOAT (I2-I7 detector)
    'mlaad_robust_goat_seed42-best-epoch=05-val-eer=0.3030.ckpt',
    'mlaad_robust_goat_seed1024-best-epoch=03-val-eer=0.2976.ckpt',
]

missing_ckpts = [f for f in REQUIRED_CHECKPOINTS if not (GOOD / f).exists()]
if missing_ckpts:
    record_failure('checkpoints not uploaded',
                   'Upload these to models/good_models/ (manual, no auto-download):\n  - '
                   + '\n  - '.join(missing_ckpts))
    print('MISSING CHECKPOINTS — upload to models/good_models/ before continuing:')
    for f in missing_ckpts:
        print('   -', f)
else:
    print(f'All {len(REQUIRED_CHECKPOINTS)} required checkpoints present in models/good_models/.')

# Historical path aliases used by original scripts. Prefer symlinks to avoid duplicate 500MB files.
# This is a list rather than a dict because one canonical checkpoint may need
# multiple historical names (for example mlaad_goat.ckpt and its best-epoch name).
ALIASES = [
    (GOOD / 'robust_goat.ckpt', MODELS / 'robust_goat.ckpt'),
    (GOOD / 'mini_goat-best-epoch=02-val-eer=0.0933.ckpt', MODELS / 'mini_goat.ckpt'),
    (GOOD / 'mlaad_goat-best-epoch=05-val-eer=0.2795.ckpt', EXP_CKPTS / 'mlaad_goat-best-epoch=05-val-eer=0.2795.ckpt'),
    (GOOD / 'mlaad_goat-best-epoch=05-val-eer=0.2795.ckpt', EXP_CKPTS / 'mlaad_goat.ckpt'),
    (GOOD / 'mlaad_robust_goat.ckpt', EXP_CKPTS / 'mlaad_robust_goat.ckpt'),
    (GOOD / 'mlaad_robust_goat_seed42-best-epoch=05-val-eer=0.3030.ckpt', EXP_CKPTS / 'mlaad_robust_goat_seed42-best-epoch=05-val-eer=0.3030.ckpt'),
    (GOOD / 'mlaad_robust_goat_seed1024-best-epoch=03-val-eer=0.2976.ckpt', EXP_CKPTS / 'mlaad_robust_goat_seed1024-best-epoch=03-val-eer=0.2976.ckpt'),
]

def link_or_copy(src: Path, dst: Path):
    if not src.exists():
        record_failure(f'checkpoint alias {rel(dst)}', f'Source checkpoint is missing: {rel(src)}')
        return False
    if dst.exists():
        return True
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.symlink(os.path.relpath(src, dst.parent), dst)
        print('symlink', rel(dst), '->', os.readlink(dst))
    except Exception:
        shutil.copy2(src, dst)
        print('copy', rel(src), '->', rel(dst))
    return True

for src, dst in ALIASES:
    link_or_copy(src, dst)

# Validate loadability of all checkpoints that are present and planned.
torch_load_orig = torch.load
def trusted_load(*a, **k):
    k.setdefault('weights_only', False)
    return torch_load_orig(*a, **k)
torch.load = trusted_load

rows = []
for p in sorted(set(list(GOOD.glob('*.ckpt')) + list(MODELS.glob('robust_goat*.ckpt')) + list(EXP_CKPTS.glob('*.ckpt')))):
    row = {'path': rel(p), 'exists': p.exists(), 'bytes': p.stat().st_size if p.exists() else 0, 'ok': False, 'error': ''}
    if p.exists():
        try:
            obj = torch.load(p, map_location='cpu')
            sd = obj.get('state_dict', obj) if isinstance(obj, dict) else obj
            row['state_tensors'] = sum(1 for v in sd.values() if hasattr(v, 'shape')) if hasattr(sd, 'values') else 0
            row['param_count'] = sum(int(v.numel()) for v in sd.values() if hasattr(v, 'numel')) if hasattr(sd, 'values') else 0
            row['sha256_12'] = sha256(p)[:12] if not p.is_symlink() else sha256(p.resolve())[:12]
            row['ok'] = row['state_tensors'] > 0 and row['param_count'] > 0
        except Exception as exc:
            row['error'] = f'{type(exc).__name__}: {exc}'
    rows.append(row)
ckpt_df = pd.DataFrame(rows)
display(ckpt_df)
failed = ckpt_df[~ckpt_df.ok]
if len(failed):
    record_failure('checkpoint load validation', failed[['path','error']].to_string(index=False))


## 3. Raw Data and Cache Bootstrap

These cells regenerate the raw/processed datasets that the experiment scripts consume. The original scripts are used wherever possible. If Hugging Face access is unavailable, the traceback is preserved in the report.


In [ ]:
if ALLOW_DOWNLOADS:
    # ASVspoof cache warmup. This creates data/asvspoof_2019_la if missing.
    run_cmd(
        'warm ASVspoof2019 Hugging Face cache',
        [sys.executable, '-c', "from datasets import load_dataset; load_dataset('Bisher/as_vspoof_2019_la', cache_dir='data/asvspoof_2019_la', trust_remote_code=True); print('ASVspoof cache ready')"],
        expected_outputs=[ROOT / 'data' / 'asvspoof_2019_la'],
    )
else:
    print('ALLOW_DOWNLOADS=False; ASVspoof cache warmup skipped.')

run_py(
    'prepare MLAAD-tiny raw snapshot and processed tensors',
    'experiments/scripts/prepare_mlaad_tiny.py',
    expected_outputs=[
        ROOT / 'experiments/data/mlaad_tiny_processed/splits/train.json',
        ROOT / 'experiments/data/mlaad_tiny_processed/splits/val.json',
        ROOT / 'experiments/data/mlaad_tiny_processed/splits/test.json',
    ],
)

run_py(
    'evaluate MLAAD checkpoints to regenerate baseline_eval JSONs',
    'experiments/scripts/eval_mlaad_baseline.py',
    expected_outputs=[
        ROOT / 'experiments/results/mlaad/baseline_eval/test_in_distribution.json',
        ROOT / 'experiments/results/mlaad/baseline_eval/test_cross_language.json',
        ROOT / 'experiments/results/mlaad/baseline_eval/summary.json',
    ],
    required_inputs=[
        ROOT / 'experiments/data/mlaad_tiny_processed/splits/test.json',
        ROOT / 'experiments/checkpoints/mlaad_goat.ckpt',
        ROOT / 'experiments/checkpoints/mlaad_robust_goat.ckpt',
    ],
)

run_py(
    'prepare mini_goat ASVspoof train/val tensors',
    'experiments/results/e_mini_goat_fusion/prepare_mini_goat_data.py',
    expected_outputs=[
        ROOT / 'experiments/data/mini_goat_processed/splits/train.json',
        ROOT / 'experiments/data/mini_goat_processed/splits/val.json',
    ],
    required_inputs=[ROOT / 'data/asvspoof_2019_la'],
)


## 4. Core Experiment Regeneration: Geometry, Caches, and Fusion

This is the dependency-ordered path for the main paper experiments. Each cell calls the original script. If an upstream cache is missing, the earlier cell generates it first.


In [ ]:
if RUN_HEAVY_EXPERIMENTS:
    run_py(
        'I2 axis production / full MLAAD wave cache / geometry battery',
        'experiments/scripts/i2_geometry_battery.py',
        expected_outputs=[
            ROOT / 'outputs/px_wave_cache/i2_full_test_waves.npz',
            ROOT / 'experiments/results/i2_geometry_battery/utt_features.csv',
            ROOT / 'experiments/results/i2_geometry_battery/system_table.csv',
            ROOT / 'experiments/results/i2_geometry_battery/i2_stats.json',
        ],
        required_inputs=[
            ROOT / 'experiments/results/mlaad/baseline_eval/test_in_distribution.json',
            ROOT / 'experiments/checkpoints/mlaad_robust_goat.ckpt',
        ],
    )

    run_py(
        'I3 position geometry and 3-seed MLAAD logits',
        'experiments/scripts/i3_position_geometry.py',
        expected_outputs=[
            ROOT / 'experiments/results/i3_position_geometry/embeddings.npz',
            ROOT / 'experiments/results/i3_position_geometry/logits_main.npy',
            ROOT / 'experiments/results/i3_position_geometry/logits_s42.npy',
            ROOT / 'experiments/results/i3_position_geometry/logits_s1024.npy',
            ROOT / 'experiments/results/i3_position_geometry/i3_stats.json',
        ],
        required_inputs=[
            ROOT / 'outputs/px_wave_cache/i2_full_test_waves.npz',
            ROOT / 'experiments/results/i2_geometry_battery/utt_features.csv',
            ROOT / 'experiments/checkpoints/mlaad_robust_goat.ckpt',
            ROOT / 'experiments/checkpoints/mlaad_robust_goat_seed42-best-epoch=05-val-eer=0.3030.ckpt',
            ROOT / 'experiments/checkpoints/mlaad_robust_goat_seed1024-best-epoch=03-val-eer=0.2976.ckpt',
        ],
    )

    run_py(
        'I7 axis fusion',
        'experiments/scripts/i7_axis_fusion.py',
        expected_outputs=[
            ROOT / 'experiments/results/i7_axis_fusion/i7_headline.csv',
            ROOT / 'experiments/results/i7_axis_fusion/i7_stats.json',
        ],
        required_inputs=[
            ROOT / 'outputs/px_wave_cache/i2_full_test_waves.npz',
            ROOT / 'experiments/results/i3_position_geometry/embeddings.npz',
            ROOT / 'experiments/results/i3_position_geometry/logits_main.npy',
            ROOT / 'experiments/results/i3_position_geometry/logits_s42.npy',
            ROOT / 'experiments/results/i3_position_geometry/logits_s1024.npy',
        ],
    )
else:
    print('RUN_HEAVY_EXPERIMENTS=False; skipped I2/I3/I7 regeneration.')


## 5. ASVspoof and ITW Regeneration

These scripts recreate ASVspoof causal/position artifacts and ITW transfer artifacts used by J3 and later audits.


In [ ]:
if RUN_HEAVY_EXPERIMENTS:
    run_py(
        'I1 ASVspoof causal decomposition and baseline logits',
        'experiments/scripts/i1_geometry_causal_decomp.py',
        expected_outputs=[
            ROOT / 'experiments/results/i1_geometry_causal_decomp/i1_logits.npz',
            ROOT / 'experiments/results/i1_geometry_causal_decomp/i1_stats.json',
        ],
        required_inputs=[
            ROOT / 'models/robust_goat.ckpt',
            ROOT / 'models/good_models/robust_goat_seed3.ckpt',
        ],
    )

    run_py(
        'I4 ASVspoof position geometry',
        'experiments/scripts/i4_asvspoof_position.py',
        expected_outputs=[
            ROOT / 'experiments/results/i4_asvspoof_position/features.npz',
            ROOT / 'experiments/results/i4_asvspoof_position/i4_stats.json',
        ],
        required_inputs=[ROOT / 'experiments/results/i1_geometry_causal_decomp/i1_logits.npz'],
    )

    run_py(
        'I6 test-time geometry boost and ITW logits',
        'experiments/scripts/i6_testtime_geometry_boost.py',
        expected_outputs=[
            ROOT / 'experiments/results/i6_testtime_boost/i6_logits.npz',
            ROOT / 'experiments/results/i6_testtime_boost/i6_results.csv',
        ],
        required_inputs=[
            ROOT / 'outputs/px_wave_cache/i2_full_test_waves.npz',
            ROOT / 'experiments/checkpoints/mlaad_robust_goat.ckpt',
        ],
    )

    run_py(
        'I5 ITW transfer',
        'experiments/scripts/i5_itw_transfer.py',
        expected_outputs=[
            ROOT / 'experiments/results/i5_itw_transfer/features.npz',
            ROOT / 'experiments/results/i5_itw_transfer/utt_table.csv',
            ROOT / 'experiments/results/i5_itw_transfer/i5_stats.json',
        ],
        required_inputs=[
            ROOT / 'experiments/results/i6_testtime_boost/i6_logits.npz',
            ROOT / 'experiments/results/i3_position_geometry/embeddings.npz',
        ],
    )
else:
    print('RUN_HEAVY_EXPERIMENTS=False; skipped I1/I4/I6/I5 regeneration.')


## 6. Paper-Level Experiments: Adaptive Axis, Prospective ASVspoof21, AASIST, mini_goat

These cells run the remaining major paper experiments through their original scripts. External AASIST assets are explicitly checked; missing assets produce a traceback and remain visible in the final report.


### AASIST baseline — how it is loaded (required by J5 & J6)

The AASIST anti-spoofing baseline is the **official implementation from
[`clovaai/aasist`](https://github.com/clovaai/aasist)**, checked out into `baselines/aasist/`.
J5 (cross-family) and J6 (MLAAD fine-tune) load it verbatim like this:

```python
sys.path.insert(0, 'baselines/aasist')
from models.AASIST import Model as AASIST                       # baselines/aasist/models/AASIST.py
conf  = json.loads(open('baselines/aasist/config/AASIST.conf').read())
model = AASIST(conf['model_config'])
model.load_state_dict(torch.load('baselines/aasist/models/weights/AASIST.pth'))  # pretrained, ~1.3 MB
```

Three files must exist — **all three ship inside the clovaai/aasist repo, including the
pretrained `AASIST.pth` weights**, so no separate weight download is needed:

| file | role |
|---|---|
| `baselines/aasist/models/AASIST.py` | model definition |
| `baselines/aasist/config/AASIST.conf` | architecture config (`model_config`) |
| `baselines/aasist/models/weights/AASIST.pth` | pretrained weights |

> ⚠️ `baselines/aasist` is a **nested git checkout, not a registered submodule** (there is no
> `.gitmodules` entry), so a fresh `git clone` of *this* repo will **not** contain it. The cell
> below clones it if the files are missing, then verifies the exact load path used by J5/J6.
> *(Added programmatically and not hand-tested end-to-end — the verify step below is your
> confirmation that it loads.)*

In [ ]:
# ── AASIST baseline setup + load verification (idempotent) ────────────────────
import shutil
AASIST_DIR   = ROOT / 'baselines' / 'aasist'
AASIST_FILES = [AASIST_DIR / 'models' / 'AASIST.py',
                AASIST_DIR / 'config' / 'AASIST.conf',
                AASIST_DIR / 'models' / 'weights' / 'AASIST.pth']

if not all(f.exists() for f in AASIST_FILES):
    print('AASIST assets missing -> cloning official clovaai/aasist ...')
    # a bare gitlink dir (from a fresh clone) would block git clone; clear it first
    if AASIST_DIR.exists() and not (AASIST_DIR / '.git').exists():
        shutil.rmtree(AASIST_DIR)
    run_cmd('clone AASIST baseline (clovaai/aasist)',
            ['git', 'clone', '--depth', '1',
             'https://github.com/clovaai/aasist.git', str(AASIST_DIR)],
            expected_outputs=AASIST_FILES)

print('AASIST required files:')
for f in AASIST_FILES:
    print(('  OK   ' if f.exists() else '  MISS ') + rel(f) +
          (f'  ({f.stat().st_size} bytes)' if f.exists() else ''))

# Verify it loads with the EXACT path J5/J6 use (subprocess, to avoid polluting
# the notebook kernel's import table).
run_cmd('verify AASIST model loads',
        [sys.executable, '-c',
         "import sys, json, torch; sys.path.insert(0, 'baselines/aasist'); "
         "from models.AASIST import Model; "
         "conf = json.load(open('baselines/aasist/config/AASIST.conf')); "
         "m = Model(conf['model_config']); "
         "m.load_state_dict(torch.load('baselines/aasist/models/weights/AASIST.pth', map_location='cpu')); "
         "print('AASIST loads OK - params:', sum(p.numel() for p in m.parameters()))"],
        required_inputs=AASIST_FILES)

In [ ]:
if RUN_HEAVY_EXPERIMENTS:
    run_py(
        'J3 axis-adaptive detector head',
        'experiments/scripts/j3_axis_adaptive_head.py',
        expected_outputs=[
            ROOT / 'experiments/results/j3_axis_adaptive/j3_results.csv',
            ROOT / 'experiments/results/j3_axis_adaptive/j3_summary.csv',
        ],
        required_inputs=[
            ROOT / 'outputs/px_wave_cache/i2_full_test_waves.npz',
            ROOT / 'experiments/results/i3_position_geometry/embeddings.npz',
            ROOT / 'experiments/results/i5_itw_transfer/features.npz',
            ROOT / 'experiments/results/i4_asvspoof_position/features.npz',
            ROOT / 'experiments/results/i1_geometry_causal_decomp/i1_logits.npz',
        ],
    )

    run_py(
        'J4 ASVspoof21 prospective/rotation check',
        'experiments/scripts/j4_asvspoof21_prospective.py',
        expected_outputs=[
            ROOT / 'experiments/results/j4_asvspoof21/j4_results.json',
            ROOT / 'experiments/results/j4_asvspoof21/j4_table.csv',
        ],
        required_inputs=[ROOT / 'experiments/results/i1_geometry_causal_decomp/i1_logits.npz'],
    )

    run_py(
        'J5 AASIST cross-family',
        'experiments/scripts/j5_aasist_crossfamily.py',
        expected_outputs=[ROOT / 'experiments/results/j5_aasist/j5_results.json'],
        required_inputs=[
            ROOT / 'baselines/aasist/models/AASIST.py',
            ROOT / 'baselines/aasist/models/weights/AASIST.pth',
            ROOT / 'baselines/aasist/config/AASIST.conf',
        ],
    )

    run_py(
        'J6 AASIST MLAAD fine-tuning',
        'experiments/scripts/j6_train_aasist_mlaad.py',
        expected_outputs=[ROOT / 'experiments/results/j6_aasist_mlaad/j6_results.json'],
        required_inputs=[
            ROOT / 'baselines/aasist/models/AASIST.py',
            ROOT / 'baselines/aasist/models/weights/AASIST.pth',
            ROOT / 'baselines/aasist/config/AASIST.conf',
            ROOT / 'experiments/data/mlaad_tiny_processed/splits/train.json',
        ],
    )

    run_py(
        'mini_goat score and axis fusion',
        'experiments/results/e_mini_goat_fusion/score_and_fuse_mini_goat.py',
        expected_outputs=[
            ROOT / 'experiments/results/e_mini_goat_fusion/mini_goat_fusion_results.json',
            ROOT / 'experiments/results/e_mini_goat_fusion/headline_comparison.csv',
        ],
        required_inputs=[
            ROOT / 'models/mini_goat.ckpt',
            ROOT / 'models/robust_goat.ckpt',
            ROOT / 'experiments/results/i1_geometry_causal_decomp/i1_logits.npz',
            ROOT / 'experiments/results/i4_asvspoof_position/features.npz',
        ],
    )
else:
    print('RUN_HEAVY_EXPERIMENTS=False; skipped J3/J4/J5/J6/mini_goat regeneration.')


## 7. Audit Suite and Paper Claim Checks

Run the original audit scripts and compare regenerated metrics against paper targets. The final table deliberately flags any mismatch instead of adjusting results.


In [ ]:
AUDIT_SCRIPTS = [
    'experiments/axis_audits/audit1_multiplicity.py',
    'experiments/axis_audits/audit2_sdalong_claim.py',
    'experiments/axis_audits/audit3_asvspoof_prospective.py',
    'experiments/axis_audits/audit4_axis_rotation.py',
    'experiments/axis_audits/audit5_fusion_claims.py',
    'experiments/axis_audits/audit6_itw_speaker.py',
    'experiments/axis_audits/audit7_agreement.py',
    'experiments/axis_audits/audit8_i1_causal.py',
    'experiments/axis_audits/audit9_hardness_reliability.py',
    'experiments/axis_audits/audit10_consistency.py',
]
for script in AUDIT_SCRIPTS:
    run_py(f'audit {Path(script).name}', script)


In [ ]:
import math
import numpy as np
import pandas as pd

checks = []
def check(name, actual, expected, tol):
    ok = actual is not None and np.isfinite(actual) and abs(float(actual) - float(expected)) <= tol
    checks.append({'metric': name, 'actual': actual, 'expected': expected, 'tol': tol, 'ok': bool(ok)})

def read_json(path):
    path = ROOT / path
    if not path.exists():
        record_failure(f'read {rel(path)}', f'Missing metric artifact: {rel(path)}')
        return {}
    return json.loads(path.read_text())

# Targets mirrored from the paper/demo audit. Keep tolerances explicit.
try:
    i7 = pd.read_csv(ROOT / 'experiments/results/i7_axis_fusion/i7_headline.csv')
    det = i7[(i7.seed == 'main') & (i7.scorer == 'detector')]['EER'].iloc[0]
    fus = i7[(i7.seed == 'main') & (i7.scorer == 'fused')]['EER'].iloc[0]
    check('I7 main-seed fused dEER', fus - det, -0.1094, 0.003)
except Exception:
    record_failure('I7 metric extraction', traceback.format_exc())

try:
    mini = read_json('experiments/results/e_mini_goat_fusion/mini_goat_fusion_results.json')
    check('mini_goat detector-alone EER', mini.get('mini_goat',{}).get('detector_alone_eer'), 0.14375, 0.002)
    check('mini_goat fused EER', mini.get('mini_goat',{}).get('fused_eer'), 0.11375, 0.002)
except Exception:
    record_failure('mini_goat metric extraction', traceback.format_exc())

try:
    j3 = pd.read_csv(ROOT / 'experiments/results/j3_axis_adaptive/j3_summary.csv')
    row = j3[(j3.dataset == 'mlaad') & (j3.head == 'lda') & (j3.n_cal == 250)]
    if len(row):
        check('J3 MLAAD LDA n=250 dEER', row['dEER'].iloc[0], -0.136055, 0.010)
except Exception:
    record_failure('J3 metric extraction', traceback.format_exc())

try:
    j5 = read_json('experiments/results/j5_aasist/j5_results.json')
    check('AASIST zero-shot MLAAD baseline EER', j5.get('eer',{}).get('mlaad'), 0.375959, 0.0005)
    check('AASIST zero-shot MLAAD fused EER', j5.get('H4',{}).get('mlaad',{}).get('EER_fused'), 0.116490, 0.0005)
except Exception:
    record_failure('J5 metric extraction', traceback.format_exc())

try:
    j6 = read_json('experiments/results/j6_aasist_mlaad/j6_results.json')
    check('AASIST fine-tuned test EER', j6.get('eer_test'), 0.200451, 0.0005)
    check('AASIST fine-tuned sd_along rho', j6.get('law',{}).get('sd_along',{}).get('rho'), 0.349498, 0.0005)
except Exception:
    record_failure('J6 metric extraction', traceback.format_exc())

try:
    j4 = read_json('experiments/results/j4_asvspoof21/j4_results.json')
    check('ASVspoof21 WavLM P3 rho', j4.get('P3',{}).get('rho'), 0.598901, 0.0005)
    check('ASVspoof21 WavLM P3 p', j4.get('P3',{}).get('p'), 0.030554, 0.0005)
except Exception:
    record_failure('J4 metric extraction', traceback.format_exc())

metric_df = pd.DataFrame(checks)
display(metric_df)
if len(metric_df) and not metric_df['ok'].all():
    record_failure('paper metric audit', metric_df[~metric_df.ok].to_string(index=False))
elif len(metric_df):
    print('All extracted paper metric checks matched targets within tolerance.')
else:
    print('No metric checks could be extracted; see failures above.')


## 8. Final Reproducibility Report

This cell summarizes every script run, every skip, and every traceback. A clean cold rerun should have no `failed`, `missing_inputs`, `missing_outputs`, or `exception` entries.


In [ ]:
run_df = pd.DataFrame(RUN_LOG)
display(run_df)
print(f'Failures recorded: {len(FAILURES)}')
for i, failure in enumerate(FAILURES, 1):
    print('\n' + '='*100)
    print(f'FAILURE {i}: {failure["step"]}')
    print('='*100)
    print(failure['traceback'][-8000:])

report = {
    'repo_root': str(ROOT),
    'force_rebuild': FORCE_REBUILD,
    'allow_downloads': ALLOW_DOWNLOADS,
    'run_heavy_experiments': RUN_HEAVY_EXPERIMENTS,
    'run_log': RUN_LOG,
    'failures': FAILURES,
}
out = ROOT / 'experiments/results/reproducibility_notebook_report.json'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(report, indent=2, default=str))
print('Wrote', rel(out))
if FAILURES:
    print('Reproducibility status: INCOMPLETE. See tracebacks above and report JSON.')
else:
    print('Reproducibility status: COMPLETE. All requested regeneration/audit steps succeeded.')


## 9. (Optional) In-the-Wild dataset — unblocks I5, I6, Audit4

These experiments were skipped in the main run because the **In-the-Wild (ITW)** speech
anti-spoofing corpus was not present. Running the cell below downloads it and then runs the
ITW-dependent steps.

**What it is / where it comes from**
- HuggingFace dataset **`mueller91/In-The-Wild`** (~31k utterances, bona-fide vs. spoof, real-world speakers).
- You may need to **accept the dataset terms** on its HF page and export a valid `HF_TOKEN`.
- `datasets.load_dataset(..., cache_dir='data/in_the_wild')` extracts `release_in_the_wild/`
  (including `meta.csv`) to the exact path `px_common.META_CSV` expects:
  `data/in_the_wild/downloads/extracted/<hash>/release_in_the_wild/meta.csv`.
- On first use, `px_common.load_itw_waves()` builds a balanced 48k-sample wave cache under `outputs/px_wave_cache/`.

**Unblocks:** I6 (test-time geometry boost + ITW logits), I5 (ITW transfer), Audit4 (axis rotation).
After it runs, re-run the **§8 Final Reproducibility Report** cell to fold these into the summary.

In [ ]:
# ── OPTIONAL: In-the-Wild (ITW) ───────────────────────────────────────────────
# Downloads mueller91/In-The-Wild and runs the ITW-dependent experiments.
ITW_META = (ROOT / 'data/in_the_wild/downloads/extracted'
            / 'c3c93f2f54ac2d261fa7010629351505bd6e05597ea22fd4a35c92dda590a3bf'
            / 'release_in_the_wild' / 'meta.csv')

run_cmd(
    'download In-the-Wild dataset',
    [sys.executable, '-c',
     "import os; from datasets import load_dataset; "
     "load_dataset('mueller91/In-The-Wild', cache_dir='data/in_the_wild', "
     "token=os.environ.get('HF_TOKEN')); print('ITW ready')"],
    expected_outputs=[ITW_META],
)

run_py(
    'I6 test-time geometry boost and ITW logits',
    'experiments/scripts/i6_testtime_geometry_boost.py',
    expected_outputs=[
        ROOT / 'experiments/results/i6_testtime_boost/i6_logits.npz',
        ROOT / 'experiments/results/i6_testtime_boost/i6_results.csv',
    ],
    required_inputs=[
        ROOT / 'outputs/px_wave_cache/i2_full_test_waves.npz',
        ROOT / 'experiments/checkpoints/mlaad_robust_goat.ckpt',
        ITW_META,
    ],
)
run_py(
    'I5 ITW transfer',
    'experiments/scripts/i5_itw_transfer.py',
    expected_outputs=[
        ROOT / 'experiments/results/i5_itw_transfer/features.npz',
        ROOT / 'experiments/results/i5_itw_transfer/utt_table.csv',
        ROOT / 'experiments/results/i5_itw_transfer/i5_stats.json',
    ],
    required_inputs=[
        ROOT / 'experiments/results/i6_testtime_boost/i6_logits.npz',
        ROOT / 'experiments/results/i3_position_geometry/embeddings.npz',
    ],
)
run_py('audit audit4_axis_rotation.py', 'experiments/axis_audits/audit4_axis_rotation.py',
       required_inputs=[ROOT / 'experiments/results/i5_itw_transfer/features.npz'])

## 10. (Optional) ASVspoof 2021 LA — unblocks J4, J5, Audit3

These were skipped because the **ASVspoof 2021 LA** evaluation keys were absent. Two inputs are needed:

1. **Audio** — HuggingFace dataset **`SpeechAntiSpoofingBenchmarks/ASVspoof2021_LA`**
   (parquet shards). `j4` fetches the first few shards automatically via `huggingface_hub`; no manual step.
2. **Official CM keys / metadata** — the file **`/tmp/keys/LA/CM/trial_metadata.txt`**
   (columns: `spk utt codec tx attack key trim phase`; `j4` filters to `codec == "none"`, the clean condition).

**How to get the keys** (from the ASVspoof 2021 organizers — the audio repo does not ship them):
```bash
# See "Keys & metadata" on https://www.asvspoof.org/index2021.html  (file: LA-keys-full.tar.gz;
# a Zenodo mirror also exists). Then:
KEYS_URL="<paste the LA-keys-full.tar.gz download URL>"
wget -O /tmp/LA-keys-full.tar.gz "$KEYS_URL"
mkdir -p /tmp/keys && tar xzf /tmp/LA-keys-full.tar.gz -C /tmp/keys
test -f /tmp/keys/LA/CM/trial_metadata.txt && echo OK
```

**Unblocks:** J4 (prospective hardness on ASVspoof2021), J5 (AASIST cross-family), Audit3 (prospective).
Requires `i1_logits.npz` (produced by the I1 cell above). Re-run the **§8** report cell afterward.

In [ ]:
# ── OPTIONAL: ASVspoof 2021 LA (clean condition) ──────────────────────────────
# Requires /tmp/keys/LA/CM/trial_metadata.txt (see the markdown above for the download).
KEYS = Path('/tmp/keys/LA/CM/trial_metadata.txt')
if not KEYS.exists():
    record_failure(
        'ASVspoof2021 keys missing',
        f'{KEYS} not found. Download LA-keys-full.tar.gz from the ASVspoof 2021 organizers '
        '(https://www.asvspoof.org/index2021.html) and extract it to /tmp/keys, then re-run this cell.')
else:
    run_py(
        'J4 ASVspoof21 prospective/rotation check',
        'experiments/scripts/j4_asvspoof21_prospective.py',
        expected_outputs=[
            ROOT / 'experiments/results/j4_asvspoof21/j4_results.json',
            ROOT / 'experiments/results/j4_asvspoof21/j4_table.csv',
        ],
        required_inputs=[
            KEYS,
            ROOT / 'experiments/results/i1_geometry_causal_decomp/i1_logits.npz',
        ],
    )
    run_py(
        'J5 AASIST cross-family',
        'experiments/scripts/j5_aasist_crossfamily.py',
        expected_outputs=[ROOT / 'experiments/results/j5_aasist/j5_results.json'],
        required_inputs=[
            ROOT / 'baselines/aasist/models/AASIST.py',
            ROOT / 'baselines/aasist/models/weights/AASIST.pth',
            ROOT / 'baselines/aasist/config/AASIST.conf',
        ],
    )
    run_py('audit audit3_asvspoof_prospective.py',
           'experiments/axis_audits/audit3_asvspoof_prospective.py',
           required_inputs=[ROOT / 'experiments/results/j4_asvspoof21/waves_sel.npz'])